In [ ]:
import torch
import torch.nn as nn
from dataclasses import dataclass


class SiLU(nn.SiLU):
    @property
    def output_multiplier(self) -> float:
        return 1.0
    # end
# end


@dataclass
class SimpleMLPConfig:
    dim_hidden: int
    dim_in: int
    dim_out: int
    bias: bool
    activation: nn.Module
# end


class SimpleMLP(nn.Module):
    def __init__(self, config_mlp):
        super().__init__()

        self.dim_hidden = config_mlp.dim_hidden
        self.dim_in = config_mlp.dim_in
        self.dim_out = config_mlp.dim_out
        self.bias = config_mlp.bias

        self.project_gate = nn.Linear(self.dim_in, self.dim_hidden, bias=self.bias)
        self.project_up = nn.Linear(self.dim_in, self.dim_hidden, bias=self.bias)
        self.project_down = nn.Linear(self.dim_hidden, self.dim_out, bias=self.bias)
        self.activation = config_mlp.activation
    # end

    def forward(self, x):
        return self.project_down(self.activation(self.project_gate(x)) * self.project_up(x))
    # end
# end


In [ ]:
config_mlp1 = SimpleMLPConfig(
    dim_in=3,
    dim_hidden=32,
    dim_out=32
    bias=True,
    activation=SiLU()
)

config_mlp2 = SimpleMLPConfig(
    dim_in=32,
    dim_hidden=32,
    dim_out=1
    bias=True,
    activation=SiLU()
)

evaluator = nn.Sequential(
    SimpleMLP(config_mlp1),
    SimpleMLP(config_mlp2)
)


In [12]:
T = L = 64
id_batch = 0
id_blk = 0
folder_base = f'../stats/{id_batch}'
pos_root = 32

pos_base = pos_root + id_blk * L
margin = torch.load(f'{folder_base}/margin_{pos_base}_{pos_base + L}.pt')
print(margin.shape)

torch.Size([64, 64])
